# RHU question classification · V3.1 FINAL (capstone classification baseline)

**CCEE 566 | RHU voice assistant | Question classification component**

This notebook retains the V2 experiment (normalization ablation, BoW/TF-IDF × Naive Bayes/Logistic Regression/Linear SVM, validation-based model selection, train+validation refit) and repairs the shortcomings identified in the V1/V2 reviews. It is **not** a complete question-answering or ABSA model.

**V3.1 scope:** finish the classifier for a one-month academic prototype. Retain all V3 model settings and data partitions, audit the official taxonomy without modifying labels, and add a QA handoff interface that returns soft routing hints, not asserted answers. The original test is a historical development benchmark.

**Changes:** (1) validate split/group integrity; (2) distinguish known-label and all-test-label metrics; (3) report label coverage and independent-group support; (4) select a category/intent consistency policy **on validation**, not test; (5) report group-weighted accuracy and template similarity; (6) export a portable inference helper and exact mapping; (7) make independent student testing optional, not fictional.

**Evaluation caveat:** the original test split was already inspected during V1/V2 development. Treat its results as a **previously consulted development benchmark**, not a pristine final holdout. A fresh, independently authored and human-labeled student set is necessary for credible final generalization claims. The notebook cannot repair absent intent training groups by changing code alone.

**Running:** Upload either the supplied **RHU QA EDA Training Package ZIP** (preferred, contains `clean_data/intent_*.csv`) or the original final-data ZIP (`INTENT_*.csv`). Locally, place the ZIP in the same folder as the notebook; Google Colab prompts for upload only if it cannot find the files. Runtime is CPU-friendly but the full original comparison grid can take several minutes. Generated files go to `rhu_v31_outputs/`. No network access or API keys are required.

## 0. Experimental protocol

1. Only **train** and **validation** determine preprocessing and model hyperparameters. Missing labels in validation are counted as errors in all-row metrics but cannot be predicted by a train-only model.
2. Select one classification family per target, using validation macro F1 on known validation labels (the original V2 decision rule). For the routing rule, compare three consistency policies on the **same validation predictions**, then lock the policy.
3. Refit selected models on train plus validation, build the intent-to-category mapping from train plus validation, and evaluate the unchanged historical test set. Do not revise rules based on the test outcomes.
4. Report accuracy, observed-test-label macro F1, known-label macro F1, missing label support, group-weighted accuracy, and invalid pairing rate. Training rows often share groups; some groups contain several intents and do not represent one intent each.
5. **Do not** use a prediction confidence threshold: uncalibrated margins are not probabilities. A verified unknown-question rejection system requires independent out-of-scope validation data.

In [1]:
from __future__ import annotations
import os, io, re, json, sys, glob, time, zipfile, hashlib, warnings, platform
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import joblib, sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
OUT_DIR = Path('rhu_v31_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
PREP = 'text_norm'                 # fixed based on V2 validation ablation; preserve comparator
STUDENT_FILE = Path('student_questions.csv')
TARGETS = ('category', 'intent')
FAMILIES = ('Naive Bayes', 'Logistic Regression', 'SVM (LinearSVC)')
EXCLUDED_PROVENANCE_CATEGORY = 'human_reviewed_catalog_fact'
T0 = time.perf_counter()
print('Python', platform.python_version(), '| sklearn', sklearn.__version__, '| pandas', pd.__version__,
      '| numpy', np.__version__, '| joblib', joblib.__version__)
print('Outputs:', OUT_DIR.resolve())

Python 3.13.5 | sklearn 1.8.0 | pandas 2.2.3 | numpy 2.3.5 | joblib 1.5.3
Outputs: /mnt/data/rhu_v31_outputs


## 1. Load the supplied final data without modifying the official partitions

`clean_data/intent_{train,validation,test}.csv` from the EDA package is preferred. The original final-data archive is accepted when that package is unavailable. A SHA-256 digest records which release was used. ZIP files are read in memory, not blindly extracted.

In [2]:
def locate_data():
    env = os.environ.get('RHU_DATA_PATH', '').strip()
    candidates = ([Path(env)] if env else [])
    for root in (Path.cwd(), Path('/mnt/data')):
        if root.is_dir():
            candidates += sorted(root.glob('*EDA*Training*Package*.zip'))
            candidates += sorted(root.glob('RHU_QA_EDA*.zip'))
            candidates += sorted(root.glob('*Train_Test_Final*.zip'))
            candidates += [root / 'clean_data']
    for p in candidates:
        if p.is_dir() and (p / 'intent_train.csv').is_file():
            return p
        if p.is_file() and p.suffix.lower() == '.zip':
            with zipfile.ZipFile(p) as z:
                names = set(z.namelist())
                if 'clean_data/intent_train.csv' in names or 'INTENT_TRAIN.csv' in names:
                    return p
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('No classification data found. Place either supplied ZIP next to notebook, or set RHU_DATA_PATH.')
    print('Upload the supplied RHU QA EDA Training Package ZIP (preferred), or final training ZIP.')
    uploaded = files.upload()
    for name in uploaded:
        p = Path(name)
        if p.suffix.lower() == '.zip':
            with zipfile.ZipFile(p) as z:
                names = set(z.namelist())
                if 'clean_data/intent_train.csv' in names or 'INTENT_TRAIN.csv' in names:
                    return p
    raise FileNotFoundError('Uploaded files did not contain the required intent split CSVs.')

DATA_SOURCE = locate_data()
IS_CLEAN = False
if DATA_SOURCE.is_dir():
    def raw_bytes(split): return (DATA_SOURCE / f'intent_{split}.csv').read_bytes()
    source_name = 'EDA clean_data folder'
    IS_CLEAN = True
else:
    with zipfile.ZipFile(DATA_SOURCE) as z:
        IS_CLEAN = 'clean_data/intent_train.csv' in z.namelist()
    def raw_bytes(split):
        member = f'clean_data/intent_{split}.csv' if IS_CLEAN else f'INTENT_{split.upper()}.csv'
        with zipfile.ZipFile(DATA_SOURCE) as z:
            return z.read(member)
    source_name = 'EDA cleaned archive' if IS_CLEAN else 'Original release archive (not EDA cleaned export)'

source_hashes = {s: hashlib.sha256(raw_bytes(s)).hexdigest() for s in ('train', 'validation', 'test')}
splits = {s: pd.read_csv(io.BytesIO(raw_bytes(s)), keep_default_na=False, low_memory=False)
          for s in ('train', 'validation', 'test')}
print('Source:', DATA_SOURCE, '|', source_name)
print('Original row counts:', {s: len(d) for s, d in splits.items()})
print('SHA256 train CSV:', source_hashes['train'])
if not IS_CLEAN:
    print('CAUTION: using original release exports, not the EDA clean_data version. Note this in your report.')

Source: /mnt/data/RHU_QA_EDA_Training_Package_2026_2027(1).zip | EDA cleaned archive
Original row counts: {'train': 7251, 'validation': 1512, 'test': 1700}
SHA256 train CSV: 4bb88fd1a2490c40797377336207a08a7391792ea25174fea7928d4c7656bc8a


## 2. Dataset and split integrity checks

Do not change the original split assignments. The 53 review-style questions whose `category` is `human_reviewed_catalog_fact` are removed from this classification experiment only; that tag is provenance, not a genuine student question category. This is **not** a judgement about their validity for other QA experiments.

In [3]:
REQUIRED = {'question_id', 'question_text', 'category', 'intent', 'group_id', 'split'}
for s, df in splits.items():
    assert REQUIRED.issubset(df.columns), f'{s} lacks columns: {REQUIRED - set(df.columns)}'
    assert (df['split'].astype(str).str.lower() == s).all(), f'Mismatched split assignment: {s}'
    assert (df[list(REQUIRED)].astype(str).apply(lambda c: c.str.strip()) != '').all().all(), f'{s} contains blank required fields'
    assert not df['question_id'].duplicated().any(), f'Duplicate question IDs within {s}'
for a,b in (('train','validation'), ('train','test'), ('validation','test')):
    for col in ('question_id', 'group_id', 'question_text'):
        overlap = set(splits[a][col]) & set(splits[b][col])
        assert not overlap, f'{a}/{b} share {col}: {list(overlap)[:3]}'

clean = {}
for s, df in splits.items():
    mask = df['category'].ne(EXCLUDED_PROVENANCE_CATEGORY)
    clean[s] = df.loc[mask].copy().reset_index(drop=True)
    print(s, ':', len(df), '->', len(clean[s]), 'classifier rows; excluded', int((~mask).sum()))
train, val, test = (clean[s] for s in ('train', 'validation', 'test'))
assert len(train) > 0 and len(val) > 0 and len(test) > 0
trainval = pd.concat([train, val], ignore_index=True)
all_rows = pd.concat([train, val, test], ignore_index=True)
assert (all_rows.groupby('group_id')['category'].nunique() == 1).all()
print('Groups containing multiple intents (legitimate in this dataset):', int((all_rows.groupby('group_id')['intent'].nunique()>1).sum()))
assert (all_rows.groupby('intent')['category'].nunique() == 1).all(), 'Intent -> category not one-to-one; taxonomy review required'
print('Distinct groups:', {s: clean[s]['group_id'].nunique() for s in clean})
print('Distinct categories:', {s: clean[s]['category'].nunique() for s in clean})
print('Distinct intents:', {s: clean[s]['intent'].nunique() for s in clean})
print('Test intent labels unavailable after train+validation:', len(set(test.intent) - set(trainval.intent)))

train : 7251 -> 7212 classifier rows; excluded 39
validation : 1512 -> 1505 classifier rows; excluded 7
test : 1700 -> 1693 classifier rows; excluded 7
Groups containing multiple intents (legitimate in this dataset): 582
Distinct groups: {'train': 466, 'validation': 103, 'test': 102}
Distinct categories: {'train': 17, 'validation': 14, 'test': 11}
Distinct intents: {'train': 78, 'validation': 31, 'test': 25}
Test intent labels unavailable after train+validation: 12


## 3. Preprocessing and representativeness diagnostics

Retain V2's three ablation variants. The classifier may mask course codes and numbers to learn question types; the original text must **never** be replaced when passing the question into evidence retrieval. Avoid stop-word removal if it removes informative question words such as *which*, *what*, and *how*.

In [4]:
PREP_STEPS = [
    (r'\b[a-z]{3,5}\s?\d{3}[a-z]?\b', ' coursecode '),
    (r'\d+', ' num '),
    (r'[^a-z\s]', ' '),
    (r'\s+', ' '),
]

def preprocess(texts, variant=PREP):
    s = pd.Series(list(texts), dtype='string').fillna('').str.lower()
    if variant == 'text_basic':
        return s.str.replace(r'[^a-z0-9\s]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip().tolist()
    for pattern, replacement in PREP_STEPS:
        s = s.str.replace(pattern, replacement, regex=True)
    s = s.str.strip()
    if variant == 'text_nostop':
        return [' '.join(w for w in t.split() if w not in ENGLISH_STOP_WORDS) for t in s.tolist()]
    if variant != 'text_norm': raise ValueError('Unknown preprocessing variant: ' + str(variant))
    return s.tolist()

for df in (train, val, test):
    for v in ('text_basic', 'text_norm', 'text_nostop'):
        df[v] = preprocess(df['question_text'], v)
# Reassemble after adding derived features: trainval was originally created from unprocessed columns.
trainval = pd.concat([train, val], ignore_index=True)

print('Preprocessing examples:')
for _, row in train.sample(4, random_state=3).iterrows():
    print('Raw:', row.question_text, '| normalized:', row.text_norm)
print('\nIndependent training groups per intent:')
counts = train.groupby('intent')['group_id'].nunique()
print('Single-group intents:', int((counts == 1).sum()), 'of', len(counts))
for target in TARGETS:
    known = set(train[target])
    print(target, '| validation labels unseen in train:', len(set(val[target]) - known),
          '| validation rows unseen:', int((~val[target].isin(known)).sum()))
print('Dominant train categories:', train.category.value_counts().head(5).to_dict())

Preprocessing examples:
Raw: What does RHU say about official title of course MATH 210? | normalized: what does rhu say about official title of course coursecode
Raw: What must I complete before enrolling in BECN 301? | normalized: what must i complete before enrolling in coursecode
Raw: What must I complete before enrolling in CIVE 240? | normalized: what must i complete before enrolling in coursecode
Raw: Give me the RHU catalog description of MECH 523. | normalized: give me the rhu catalog description of coursecode

Independent training groups per intent:
Single-group intents: 62 of 78
category | validation labels unseen in train: 0 | validation rows unseen: 0
intent | validation labels unseen in train: 14 | validation rows unseen: 138
Dominant train categories: {'courses': 4257, 'study_plans': 1811, 'academic_regulations': 254, 'tuition_fees': 231, 'admissions': 140}


## 4. Original V2 preprocessing ablation (validation only)

This preserves the comparability of the original notebook. A validation metric over **known labels** measures learnable classes; all-row accuracy counts unknown-class examples as errors. Do not use test examples to pick preprocessing.

In [5]:
abl_rows = []
for target in TARGETS:
    yva = val[target].to_numpy()
    known = val[target].isin(set(train[target])).to_numpy()
    for variant in ('text_basic', 'text_norm', 'text_nostop'):
        vec = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)
        Xtr, Xva = vec.fit_transform(train[variant]), vec.transform(val[variant])
        model = LogisticRegression(C=10, class_weight='balanced', max_iter=2000)
        with warnings.catch_warnings(record=True) as ww:
            warnings.simplefilter('always', ConvergenceWarning)
            pr = model.fit(Xtr, train[target]).predict(Xva)
        abl_rows.append({'target':target, 'preprocessing':variant, 'features':Xtr.shape[1],
             'macroF1_known_val':f1_score(yva[known], pr[known], labels=sorted(set(yva[known])), average='macro', zero_division=0),
             'accuracy_all_val':accuracy_score(yva,pr),
             'convergence_warnings':sum(issubclass(w.category,ConvergenceWarning) for w in ww)})
ablation = pd.DataFrame(abl_rows)
display(ablation.round(4))
print('Main comparison stays fixed at', PREP, 'to maintain continuity with V2.')

     target preprocessing  ...  accuracy_all_val  convergence_warnings
0  category    text_basic  ...            0.8671                     0
1  category     text_norm  ...            0.8738                     0
2  category   text_nostop  ...            0.8532                     0
3    intent    text_basic  ...            0.8093                     0
4    intent     text_norm  ...            0.8113                     0
5    intent   text_nostop  ...            0.7914                     0

[6 rows x 6 columns]
Main comparison stays fixed at text_norm to maintain continuity with V2.


## 5. Model search using validation only

Preserve V2's BoW/TF-IDF and three classical classifier families. For each family, choose its best converged configuration on validation macro F1 of **known** gold labels. The selection criterion is stated before viewing test scores; comparisons on previously viewed test results must not change it.

For a fast rerun of the **unchanged** V3 hyperparameter search, set `RHU_USE_VERIFIED_V3_CACHE=1` and place the V3 model ZIP at `/mnt/data/RHU_Question_Classification_V3_Model_Package.zip`. The notebook checks SHA-256 hashes of all three original CSVs before reusing the historical validation grid. Otherwise, it runs the complete grid itself.

In [6]:
VECTORIZERS = {
    'BoW 1-gram': ('count',(1,1)), 'BoW 1-2gram': ('count',(1,2)),
    'TFIDF 1-gram': ('tfidf',(1,1)), 'TFIDF 1-2gram': ('tfidf',(1,2))}
MODEL_GRID = []
for a in (0.1,0.5,1.0):
    MODEL_GRID.append(('Naive Bayes', f'alpha={a}', MultinomialNB(alpha=a)))
for c in (1,10,100):
    for cw in (None,'balanced'):
        MODEL_GRID.append(('Logistic Regression',f'C={c},cw={cw}',LogisticRegression(C=c,class_weight=cw,max_iter=2000)))
for c in (0.1,1):
    for cw in (None,'balanced'):
        MODEL_GRID.append(('SVM (LinearSVC)', f'C={c},cw={cw}',LinearSVC(C=c,class_weight=cw,max_iter=5000,dual=True)))

def vectorizer_for(name):
    kind, ngram = VECTORIZERS[name]
    return CountVectorizer(ngram_range=ngram) if kind == 'count' else TfidfVectorizer(ngram_range=ngram,sublinear_tf=True)

def pipeline_for(record):
    return Pipeline([('vectorizer',vectorizer_for(record['vectorizer'])),
                     ('classifier',clone(record['model']))])

# Optional fast rerun: reuse the exact V3 validation search table ONLY when
# the original training, validation, and test CSV hashes match.
# If missing or disabled, the complete original V2/V3 search is executed.
USE_VERIFIED_GRID = os.getenv('RHU_USE_VERIFIED_V3_CACHE', '0') == '1'
grid_source = 'complete search executed in this run'
if USE_VERIFIED_GRID:
    cache_file = Path('/mnt/data/RHU_Question_Classification_V3_Model_Package.zip')
    if not cache_file.exists():
        raise FileNotFoundError('Verified search cache is not available. Unset RHU_USE_VERIFIED_V3_CACHE to run full search.')
    with zipfile.ZipFile(cache_file) as cached:
        cache_manifest = json.loads(cached.read('v3_experiment_report.json'))
        assert cache_manifest['input_sha256'] == source_hashes, 'Cannot reuse grid: input dataset checksum changed'
        grid = pd.read_csv(io.BytesIO(cached.read('validation_model_grid.csv')))
    grid['converged'] = True  # Original V3 run reported no convergence failures.
    models_by_key = {(family,cfg):proto for family,cfg,proto in MODEL_GRID}
    best, grid_rows = {}, grid.to_dict(orient='records')
    for row in grid_rows:
        key=(row['target'],row['family'])
        proto=models_by_key[(row['family'],row['config'])]
        if key not in best or row['val_macroF1_known'] > best[key]['val_macroF1_known'] + 1e-12:
            best[key] = {**row,'model':proto}
    grid_source = 'verified cached V3 grid, exact dataset hashes match'
    print('Using validated V3 grid; original model and vectorizer definitions unchanged.')
else:
    best, grid_rows = {}, []
    for target in TARGETS:
        ytr, yva = train[target].to_numpy(), val[target].to_numpy()
        known = val[target].isin(set(ytr)).to_numpy()
        for vname in VECTORIZERS:
            vec = vectorizer_for(vname)
            Xtr, Xva = vec.fit_transform(train[PREP]), vec.transform(val[PREP])
            for family, cfg, proto in MODEL_GRID:
                with warnings.catch_warnings(record=True) as ww:
                    warnings.simplefilter('always', ConvergenceWarning)
                    pred = clone(proto).fit(Xtr,ytr).predict(Xva)
                converged = not any(issubclass(w.category,ConvergenceWarning) for w in ww)
                known_f1 = f1_score(yva[known],pred[known],labels=sorted(set(yva[known])),average='macro',zero_division=0)
                row = {'target':target,'vectorizer':vname,'family':family,'config':cfg,
                       'val_macroF1_known':known_f1,'val_acc_all':accuracy_score(yva,pred),'converged':converged}
                grid_rows.append(row)
                key = (target,family)
                if converged and (key not in best or known_f1 > best[key]['val_macroF1_known'] + 1e-12):
                    best[key] = {**row,'model':proto}
            print('Finished',target, vname,'| elapsed %.1fs'%(time.perf_counter()-T0),flush=True)

    grid = pd.DataFrame(grid_rows)
assert len(best) == len(TARGETS) * len(FAMILIES)
display(grid.groupby(['target','family']).agg(max_val_macroF1=('val_macroF1_known','max'),
             nonconverged=('converged',lambda x:int((~x).sum()))).round(4))
selected_families = {}
for target in TARGETS:
    opts = [best[(target,f)] for f in FAMILIES]
    winner = sorted(opts,key=lambda x:(-x['val_macroF1_known'],-x['val_acc_all'],FAMILIES.index(x['family'])))[0]
    selected_families[target] = winner['family']
    print('Selected',target,'=>',winner['family'],winner['vectorizer'],winner['config'],
          'validation known macroF1 %.4f'%winner['val_macroF1_known'])
grid.drop(columns='converged').to_csv(OUT_DIR/'validation_model_grid.csv',index=False)

Using validated V3 grid; original model and vectorizer definitions unchanged.
                              max_val_macroF1  nonconverged
target   family                                            
category Logistic Regression           0.6284             0
         Naive Bayes                   0.6242             0
         SVM (LinearSVC)               0.6745             0
intent   Logistic Regression           0.7171             0
         Naive Bayes                   0.6866             0
         SVM (LinearSVC)               0.7581             0
Selected category => SVM (LinearSVC) TFIDF 1-2gram C=1,cw=balanced validation known macroF1 0.6745
Selected intent => SVM (LinearSVC) TFIDF 1-gram C=1,cw=balanced validation known macroF1 0.7581


## 6. Detect and resolve invalid category/intent combinations (validation-based policy)

Every observed intent belongs to one category in the supplied taxonomy. Independent classifiers can contradict each other. We compare these three deterministic policies **on validation**, with models trained on **train only**:

* **category governs:** retain predicted category; within it choose the highest-scoring compatible intent if the original intent disagrees.
* **intent governs:** retain predicted intent and derive its unique category.
* **review on conflict:** retain category and withhold intent (`None`) when they disagree; no fabricated valid label.

A pair is *valid* when its intent maps to its category. A missing intent is explicitly unresolved, not an implicit prediction. We select the policy on validation using the mean of category macro F1 across validation gold labels and known-label intent macro F1. This equal-weighted score is a project design decision and is **not** a new trained classifier. Show coverage separately so abstention cannot silently inflate metrics.

In [7]:
val_pipelines = {}
for target in TARGETS:
    rec = best[(target,selected_families[target])]
    val_pipelines[target] = pipeline_for(rec).fit(train[PREP],train[target])

# Mapping must originate in the training data during policy selection.
def category_map(df):
    g = df.groupby('intent')['category'].nunique()
    if (g > 1).any(): raise ValueError('Ambiguous intent-to-category mapping: ' + ', '.join(g[g>1].index))
    return df.drop_duplicates('intent').set_index('intent')['category'].to_dict()

train_map = category_map(train)
def available_scores(model, normalized_text):
    """Uncalibrated within-class ranking scores; NEVER use as confidence probabilities."""
    vecX = model.named_steps['vectorizer'].transform(normalized_text)
    estimator = model.named_steps['classifier']
    if hasattr(estimator,'decision_function'):
        arr = np.asarray(estimator.decision_function(vecX))
    elif hasattr(estimator,'predict_log_proba'):
        arr = np.asarray(estimator.predict_log_proba(vecX))
    else:
        raise TypeError('Selected classifier cannot rank alternate intents')
    if arr.ndim == 1:
        arr = np.column_stack((-arr,arr))
    return arr, np.asarray(estimator.classes_)

def reconcile(categories, intents, score_matrix, score_classes, mapping, policy):
    categories, intents = np.asarray(categories,dtype=object),np.asarray(intents,dtype=object)
    result_cat, result_int = categories.copy(), intents.copy()
    statuses = []
    for i,(cat,intent) in enumerate(zip(categories,intents)):
        if mapping.get(intent) == cat:
            statuses.append('ok')
            continue
        if policy == 'intent_governs':
            result_cat[i] = mapping.get(intent,cat)
            result_int[i] = intent if intent in mapping else None
            statuses.append('reconciled_from_intent' if intent in mapping else 'review_missing_mapping')
        elif policy == 'category_governs':
            candidates = [j for j,x in enumerate(score_classes) if mapping.get(x) == cat]
            if candidates:
                result_int[i] = score_classes[max(candidates,key=lambda j:score_matrix[i,j])]
                statuses.append('reconciled_from_category')
            else:
                result_int[i] = None
                statuses.append('review_no_intent_for_category')
        elif policy == 'review_on_conflict':
            result_int[i] = None
            statuses.append('review_conflicting_intent')
        else: raise ValueError('Unknown policy: '+str(policy))
    # No contradictory *resolved* category/intent pair is permitted.
    assert all(i is None or mapping.get(i)==c for c,i in zip(result_cat,result_int))
    return result_cat,result_int,np.array(statuses)

def classification_metrics(y, pred, known_labels):
    y = np.asarray(y,dtype=str)
    p = np.array(['__unresolved__' if x is None else str(x) for x in pred])
    m = np.isin(y,list(known_labels))
    present = sorted(set(y))
    # Gold-present macro F1 excludes predictions on classes not actually present in gold, explicitly named.
    return {'accuracy_all':float(accuracy_score(y,p)),
            'macroF1_gold_present':float(f1_score(y,p,labels=present,average='macro',zero_division=0)),
            'macroF1_known_gold':float(f1_score(y[m],p[m],labels=sorted(set(y[m])),average='macro',zero_division=0)) if m.any() else np.nan,
            'known_rows':int(m.sum()),'unknown_rows':int((~m).sum()),
            'test_labels_present':len(present), 'labels_known_present':len(set(y[m]))}

cv = val_pipelines['category'].predict(val[PREP])
iv = val_pipelines['intent'].predict(val[PREP])
scores_v, score_classes_v = available_scores(val_pipelines['intent'],val[PREP])
print('Raw validation pair disagreements:',sum(train_map.get(i)!=c for c,i in zip(cv,iv)),'/',len(val))
policies = ('category_governs','intent_governs','review_on_conflict')
policy_rows = []
for policy in policies:
    cats,ints,statuses = reconcile(cv,iv,scores_v,score_classes_v,train_map,policy)
    cm = classification_metrics(val.category,cats,set(train.category))
    im = classification_metrics(val.intent,ints,set(train.intent))
    utility = (cm['macroF1_gold_present'] + im['macroF1_known_gold'])/2
    policy_rows.append({'policy':policy,'selection_score':utility,
       'category_macroF1_gold':cm['macroF1_gold_present'],
       'intent_macroF1_known':im['macroF1_known_gold'],
       'category_acc_all':cm['accuracy_all'],'intent_acc_all':im['accuracy_all'],
       'intent_abstentions':int(pd.isna(ints).sum()),'validation_reconciliations':int((statuses!='ok').sum())})
policy_table = pd.DataFrame(policy_rows).sort_values(['selection_score','intent_abstentions'],ascending=[False,True])
display(policy_table.round(4))
SELECTED_POLICY = str(policy_table.iloc[0].policy)
print('POLICY SELECTED ON VALIDATION ONLY:',SELECTED_POLICY)
policy_table.to_csv(OUT_DIR/'validation_routing_policies.csv',index=False)

Raw validation pair disagreements: 113 / 1505
               policy  ...  validation_reconciliations
0    category_governs  ...                         113
2  review_on_conflict  ...                         113
1      intent_governs  ...                         113

[3 rows x 8 columns]
POLICY SELECTED ON VALIDATION ONLY: category_governs


## 7. Freeze experiment and evaluate the historical test data once

Refit on train plus validation (the original V2 protocol), apply the validation-chosen policy, and print **raw vs reconciled** metrics. The test is a previously consulted benchmark across V1–V3, not pristine. Unknown gold intent labels remain errors; test examples must never enter training.

In [8]:
final_pipelines = {}
for target in TARGETS:
    rec = best[(target,selected_families[target])]
    final_pipelines[target] = pipeline_for(rec).fit(trainval[PREP],trainval[target])
final_map = category_map(trainval)

raw_cat = final_pipelines['category'].predict(test[PREP])
raw_int = final_pipelines['intent'].predict(test[PREP])
test_scores, test_classes = available_scores(final_pipelines['intent'],test[PREP])
final_cat, final_int, final_status = reconcile(raw_cat,raw_int,test_scores,test_classes,final_map,SELECTED_POLICY)

rows=[]
for state, cp, ip in [('raw',raw_cat,raw_int),('routed',final_cat,final_int)]:
    for target, pred in [('category',cp),('intent',ip)]:
        y = test[target]
        m = classification_metrics(y,pred,set(trainval[target]))
        rows.append({'stage':state,'target':target,**m})
metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))
print('Raw invalid category/intent pairs:',int(sum(final_map.get(i)!=c for c,i in zip(raw_cat,raw_int))))
print('Routed invalid resolved pairs:',int(sum(i is not None and final_map.get(i)!=c for c,i in zip(final_cat,final_int))))
print('Routed unresolved intent rows:',int(pd.isna(final_int).sum()))

predictions = test[['question_id','group_id','question_text','category','intent']].copy()
predictions = predictions.rename(columns={'category':'gold_category','intent':'gold_intent'})
predictions['raw_category'] = raw_cat
predictions['raw_intent'] = raw_int
predictions['category_pred'] = final_cat
predictions['intent_pred'] = final_int
predictions['routing_status'] = final_status
predictions['category_correct'] = predictions.gold_category == predictions.category_pred
predictions['intent_correct'] = predictions.gold_intent == predictions.intent_pred
predictions['gold_intent_supported'] = predictions.gold_intent.isin(set(trainval.intent))
predictions.to_csv(OUT_DIR/'historical_test_predictions.csv',index=False)
metrics_df.to_csv(OUT_DIR/'historical_test_metrics.csv',index=False)

    stage    target  ...  test_labels_present  labels_known_present
0     raw  category  ...                   11                    11
1     raw    intent  ...                   25                    13
2  routed  category  ...                   11                    11
3  routed    intent  ...                   25                    13

[4 rows x 9 columns]
Raw invalid category/intent pairs: 206
Routed invalid resolved pairs: 0
Routed unresolved intent rows: 0


## 8. Coverage, group-weighted metrics, and diagnostic errors

Accuracy at question level gives disproportionate weight to groups with many paraphrases. **Group-weighted accuracy** averages correctness across groups, but remains a diagnostic, not an unbiased student population estimate. A group-level bootstrap interval is included for descriptive uncertainty; it cannot correct generated-template sampling bias.

In [9]:
def group_accuracy(df, correct_col):
    gp = df.groupby('group_id')[correct_col].mean().to_numpy(dtype=float)
    avg = float(gp.mean())
    rng = np.random.default_rng(RANDOM_SEED)
    boots = gp[rng.integers(0,len(gp),size=(1200,len(gp)))].mean(axis=1)
    return {'group_weighted_accuracy':avg, 'groups':len(gp),
            'group_bootstrap_CI_95_low':float(np.quantile(boots,0.025)),
            'group_bootstrap_CI_95_high':float(np.quantile(boots,0.975))}
for target in TARGETS:
    target_metric = group_accuracy(predictions,target+'_correct')
    print(target,'group-weighted diagnostics:',{k:round(v,4) if isinstance(v,float) else v for k,v in target_metric.items()})
    bylab = predictions.groupby('gold_'+target).agg(
        test_rows=('question_id','size'),test_groups=('group_id','nunique'),
        recall=(target+'_correct','mean'))
    bylab['trainval_groups'] = trainval.groupby(target).group_id.nunique().reindex(bylab.index).fillna(0).astype(int)
    bylab['trainval_rows'] = trainval[target].value_counts().reindex(bylab.index).fillna(0).astype(int)
    bylab['support_status'] = np.where(bylab.trainval_rows.eq(0),'UNSEEN',np.where(bylab.trainval_groups.le(1),'ONE_GROUP','MULTI_GROUP'))
    bylab.sort_values(['support_status','test_rows'],ascending=[True,False]).to_csv(OUT_DIR/f'{target}_test_coverage.csv')
    print('\n',target,'labels in test with absent/one-group trainval support (NO test-support threshold):')
    print(bylab[bylab.support_status.ne('MULTI_GROUP')].sort_values('test_rows',ascending=False).round(3).to_string())

major_categories = train.category.value_counts().head(2).index.tolist()  # fixed from TRAIN distribution only
for subset_name,mask in [('dominant_train_categories',test.category.isin(major_categories)),
                         ('other_categories',~test.category.isin(major_categories))]:
    print(subset_name,'n=',int(mask.sum()),'category_accuracy=',round(predictions.loc[mask,'category_correct'].mean(),4),
          'intent_accuracy=',round(predictions.loc[mask,'intent_correct'].mean(),4))

train_norm = set(train.text_norm)
twins = test.text_norm.isin(train_norm)
print('Normalized questions identical to train:',int(twins.sum()),'/',len(test))
for label,mask in [('normalized twin',twins),('no normalized twin',~twins)]:
    print(label,'n=',int(mask.sum()),'category acc',round(predictions.loc[mask,'category_correct'].mean(),4),
          'intent acc',round(predictions.loc[mask,'intent_correct'].mean(),4))

for target in TARGETS:
    bad = predictions.loc[~predictions[target+'_correct']]
    print('\nTOP',target,'confusions:')
    print(bad.groupby(['gold_'+target,target+'_pred'],dropna=False).size().sort_values(ascending=False).head(12).to_string())

category group-weighted diagnostics: {'group_weighted_accuracy': 0.9398, 'groups': 102, 'group_bootstrap_CI_95_low': 0.8964, 'group_bootstrap_CI_95_high': 0.9779}

 category labels in test with absent/one-group trainval support (NO test-support threshold):
Empty DataFrame
Columns: [test_rows, test_groups, recall, trainval_groups, trainval_rows, support_status]
Index: []
intent group-weighted diagnostics: {'group_weighted_accuracy': 0.8846, 'groups': 102, 'group_bootstrap_CI_95_low': 0.8202, 'group_bootstrap_CI_95_high': 0.9371}

 intent labels in test with absent/one-group trainval support (NO test-support threshold):
                                    test_rows  test_groups  recall  trainval_groups  trainval_rows support_status
gold_intent                                                                                                      
program_credits                           113            1   0.000                0              0         UNSEEN
program_degree                  

## 9. Taxonomy and source-data limitations

The notebook cannot safely merge labels without a documented taxonomy decision and new ground-truth review. Related names such as `course_credit`, `course_credits`, `program_credits`, and `study_plan_credits` may have different semantics. The team must adjudicate any merges on **training-side data**, then create a new benchmark. Never use the test labels as a source for merges.

In [10]:
print('All observed category labels (including test, documentation only):',len(set(all_rows.category)))
print('All observed intent labels (including test, documentation only):',len(set(all_rows.intent)))
print('Intents trained on only one independent group:',int((train.groupby('intent').group_id.nunique() == 1).sum()))
print('Test intents absent from train+val:',sorted(set(test.intent)-set(trainval.intent)))
print('Possible labels for taxonomy review (NO AUTOMATIC MERGE):')
for token in ('credit','fee','prerequisite'):
    print(token,':',sorted(x for x in set(all_rows.intent) if token in x))

All observed category labels (including test, documentation only): 17
All observed intent labels (including test, documentation only): 104
Intents trained on only one independent group: 62
Test intents absent from train+val: ['clubs', 'counseling', 'double_major', 'freshman_aptitude', 'grade_appeal', 'petitions', 'program_accreditation', 'program_credits', 'program_degree', 'program_status', 'unanswerable_personal_aid_decision', 'work_study']
Possible labels for taxonomy review (NO AUTOMATIC MERGE):
credit : ['course_credit', 'course_credits', 'minor_credits', 'minor_unique_credits', 'program_accreditation', 'program_credits', 'study_plan_credits', 'transfer_credit']
fee : ['fee_amount', 'midterm_feedback', 'published_fee']
prerequisite : ['course_prerequisite', 'study_plan_prerequisite']


## 9A. Audit the official RHU label taxonomy (without altering labels)

The original release supplies `RHU_LABEL_TAXONOMY.csv`. Compare it to the **training + validation mapping** already used by V3 to ensure that every trained intent has a declared parent. Do **not** add test-only labels to the model from this audit. The release taxonomy can list one descriptive intent name under several categories; this is distinct from the unambiguous mapping among currently trained labels. Save the exact trained mapping for the chatbot team.

In [11]:
def load_release_taxonomy():
    # Preferred: nested release ZIP already included inside the supplied EDA package.
    archives = [DATA_SOURCE] if DATA_SOURCE.is_file() else []
    archives += [p for p in (Path.cwd(), Path('/mnt/data')) for p in Path(p).glob('*Train_Test_Final*.zip')]
    seen = set()
    for p in archives:
        if str(p) in seen or not p.is_file():
            continue
        seen.add(str(p))
        with zipfile.ZipFile(p) as outer:
            names = set(outer.namelist())
            if 'RHU_LABEL_TAXONOMY.csv' in names:
                return pd.read_csv(io.BytesIO(outer.read('RHU_LABEL_TAXONOMY.csv')), keep_default_na=False), str(p)
            nested = next((s for s in outer.namelist() if s.endswith('RHU_QA_Train_Test_Final_2026_2027.zip')),None)
            if nested:
                with zipfile.ZipFile(io.BytesIO(outer.read(nested))) as inner:
                    if 'RHU_LABEL_TAXONOMY.csv' in inner.namelist():
                        return pd.read_csv(io.BytesIO(inner.read('RHU_LABEL_TAXONOMY.csv')), keep_default_na=False), str(p)+'::'+nested
    return None, None

release_taxonomy, taxonomy_source = load_release_taxonomy()
trained_mapping = category_map(trainval)
map_frame = pd.DataFrame([{'intent':k,'category':v,
    'train_rows':int((train.intent==k).sum()),'train_groups':int(train.loc[train.intent.eq(k),'group_id'].nunique()),
    'validation_rows':int((val.intent==k).sum())} for k,v in sorted(trained_mapping.items())])
map_frame.to_csv(OUT_DIR/'documented_trained_intent_taxonomy.csv',index=False)
assert trained_mapping == final_map, 'The inference mapping must match the documented trained mapping'
print('Documented trained intents:',len(map_frame),'| trained categories:',map_frame.category.nunique())

if release_taxonomy is None:
    print('Source taxonomy not available; documented mapping derived only from train + validation.')
    multi_parent_source_labels = []
    taxonomy_audit_status = 'not_available'
else:
    assert {'label_type','label','parent'}.issubset(release_taxonomy.columns)
    source_intents = release_taxonomy.loc[release_taxonomy.label_type.eq('intent')]
    source_parents = source_intents.groupby('label')['parent'].agg(lambda xs:set(xs))
    multi_parent_source_labels = sorted(k for k,parents in source_parents.items() if len(parents)>1)
    missing_source_intents = sorted(set(trained_mapping)-set(source_parents.index))
    parent_disagreements = sorted(k for k,v in trained_mapping.items() if k in source_parents.index and v not in source_parents[k])
    assert not missing_source_intents, f'Trained intents missing from official taxonomy: {missing_source_intents}'
    assert not parent_disagreements, f'Trained mapping differs from source taxonomy: {parent_disagreements}'
    taxonomy_audit_status = 'pass'
    print('Official source:',taxonomy_source)
    print('Official taxonomy intent labels:',len(source_parents),'| rows:',len(source_intents))
    print('Source labels with multiple category parents:',multi_parent_source_labels)
    print('Trained intent-parent discrepancies:',len(parent_disagreements))

# This is a documentation audit only. Model selection and trained classes stay frozen.

Documented trained intents: 92 | trained categories: 17
Official source: /mnt/data/RHU_QA_EDA_Training_Package_2026_2027(1).zip::data/RHU_QA_Train_Test_Final_2026_2027.zip
Official taxonomy intent labels: 111 | rows: 122
Source labels with multiple category parents: ['catalog_section_explanation']
Trained intent-parent discrepancies: 0


## 10. Optional independent student-question evaluation

**No synthetic or model-generated answers will be called gold labels.** Create `student_questions.csv` with columns `question_text,category,intent`, and add genuinely independent, human-labeled RHU student questions. Keep the file out of training and validation. Report unsupported labels separately. A blank CSV template is created automatically.

In [12]:
(OUT_DIR/'student_questions_template.csv').write_text('question_text,category,intent\n',encoding='utf-8')
if not STUDENT_FILE.exists():
    print('No independently authored student_questions.csv supplied. External evaluation: NOT PERFORMED.')
    print('Fill the blank template at',OUT_DIR/'student_questions_template.csv','and place completed CSV next to notebook.')
else:
    student = pd.read_csv(STUDENT_FILE,keep_default_na=False)
    assert {'question_text','category','intent'}.issubset(student.columns)
    assert len(student)>0 and student[['question_text','category','intent']].astype(str).ne('').all().all()
    student_norm = preprocess(student.question_text)
    sc = final_pipelines['category'].predict(student_norm)
    si = final_pipelines['intent'].predict(student_norm)
    ss, cls = available_scores(final_pipelines['intent'],student_norm)
    rc,ri,st = reconcile(sc,si,ss,cls,final_map,SELECTED_POLICY)
    erows=[]
    for target,pr in [('category',rc),('intent',ri)]:
        met=classification_metrics(student[target],pr,set(trainval[target]))
        erows.append({'target':target,**met})
    print('Independent student-question metrics (ONLY if manually verified):')
    display(pd.DataFrame(erows).round(4))
    student.assign(predicted_category=rc,predicted_intent=ri,routing_status=st).to_csv(
        OUT_DIR/'independent_student_predictions.csv',index=False)

No independently authored student_questions.csv supplied. External evaluation: NOT PERFORMED.
Fill the blank template at rhu_v31_outputs/student_questions_template.csv and place completed CSV next to notebook.


## 11. Export model and reproducible standalone inference API

The Joblib bundle contains **standard scikit-learn pipelines** (vectorizer + estimator), explicit regex normalization rules, and the unique intent/category map. A separate standard-Python helper accepts raw questions, keeps them unchanged for retrieval, validates input, applies the selected routing rule, and emits review flags. There is **no calibrated probability** and `review_needed=False` is **not** a correctness guarantee.

Only load Joblib bundles from trusted sources. Serialization compatibility across scikit-learn versions is not guaranteed; the tested environment is recorded.

In [13]:
API_SOURCE = r"""# Portable RHU V3.1 classifier inference. Needs pandas, numpy, scikit-learn, joblib.
# Only load the accompanying joblib file from a trusted source.
import re
import numpy as np
import joblib


def normalize_questions(texts, regex_steps):
    out = []
    for text in texts:
        s = str(text).lower()
        for pat,rep in regex_steps:
            s = re.sub(pat,rep,s)
        out.append(s.strip())
    return out


def _scores(model, normalized):
    x = model.named_steps['vectorizer'].transform(normalized)
    estimator = model.named_steps['classifier']
    if hasattr(estimator,'decision_function'):
        scores = np.asarray(estimator.decision_function(x))
    elif hasattr(estimator,'predict_log_proba'):
        scores = np.asarray(estimator.predict_log_proba(x))
    else:
        raise TypeError('Classifier cannot rank alternate intent classes')
    if scores.ndim == 1:
        scores = np.column_stack([-scores,scores])
    return scores,np.asarray(estimator.classes_)


def load_bundle(path='rhu_classifier_v31.joblib'):
    return joblib.load(path) # trusted local artifact ONLY


def predict_questions(bundle, questions):
    if isinstance(questions,str):
        questions = [questions]
    else:
        questions = list(questions)
    if any(not isinstance(q,str) for q in questions):
        raise TypeError('Each input question must be a string')
    results = [{'question_text':q,'predicted_category':None,'predicted_intent':None,
                'raw_category':None,'raw_intent':None,'routing_status':'invalid_input',
                'review_needed':True} for q in questions]
    valid_idx = [i for i,q in enumerate(questions) if q.strip() and re.search(r'[A-Za-z]',q)]
    if not valid_idx:
        return results
    normalized = normalize_questions([questions[i] for i in valid_idx],bundle['regex_steps'])
    cats = bundle['models']['category'].predict(normalized)
    ints = bundle['models']['intent'].predict(normalized)
    scores,classes = _scores(bundle['models']['intent'],normalized)
    mapping=bundle['intent_to_category']
    policy=bundle['routing_policy']
    for j,i in enumerate(valid_idx):
        cat, intent = str(cats[j]),str(ints[j])
        raw_cat,raw_int = cat,intent
        status='ok'
        if mapping.get(intent)!=cat:
            if policy=='intent_governs':
                if intent in mapping:
                    cat=mapping[intent]
                    status='reconciled_from_intent'
                else:
                    intent=None
                    status='review_missing_mapping'
            elif policy=='category_governs':
                candidates = [k for k,x in enumerate(classes) if mapping.get(str(x))==cat]
                if candidates:
                    intent=str(classes[max(candidates,key=lambda k:scores[j,k])])
                    status='reconciled_from_category'
                else:
                    intent=None
                    status='review_no_intent_for_category'
            elif policy=='review_on_conflict':
                intent=None
                status='review_conflicting_intent'
            else:
                raise ValueError('Unexpected routing policy')
        assert intent is None or mapping[intent]==cat
        results[i]={'question_text':questions[i], 'predicted_category':cat,
            'predicted_intent':intent,'raw_category':raw_cat,'raw_intent':raw_int,
            'routing_status':status,'review_needed':status!='ok'}
    return results


def predict_one(bundle,question):
    return predict_questions(bundle,[question])[0]


def make_qa_request(prediction):
    # Soft routing hints only. The retrieval module must independently verify evidence and cite pages.
    if not isinstance(prediction,dict) or 'question_text' not in prediction:
        raise TypeError('Provide one predict_one/predict_questions result dictionary')
    valid = prediction.get('routing_status') != 'invalid_input'
    return {
        'original_question': prediction['question_text'],
        'input_valid': bool(valid),
        'category_hint': prediction.get('predicted_category') if valid else None,
        'intent_hint': prediction.get('predicted_intent') if valid else None,
        'routing_status': prediction.get('routing_status'),
        'routing_review_needed': bool(prediction.get('review_needed')),
        'category_hard_filter': False,
        'intent_hard_filter': False,
        'retrieve_from_full_catalog': bool(valid),
        'requires_verified_evidence': True,
        'requires_source_citation': True,
        'fallback_if_no_evidence': True,
    }
"""
api_path = OUT_DIR/'rhu_classifier_api.py'
api_path.write_text(API_SOURCE,encoding='utf-8')

bundle = {'version':'3.1','models':final_pipelines,'intent_to_category':final_map,
          'routing_policy':SELECTED_POLICY,'regex_steps':PREP_STEPS,
          'selected_families':selected_families,
          'trainval_rows':len(trainval),'training_data_sha256':source_hashes,
          'sklearn_version':sklearn.__version__,
          'notes':'Only classifier metadata. Preserve ORIGINAL question for evidence retrieval. Margins are not probabilities.'}
model_path=OUT_DIR/'rhu_classifier_v31.joblib'
joblib.dump(bundle,model_path)

# Check that the API runs against a fresh reloaded model, including blanks and course-code preservation.
import importlib.util
spec=importlib.util.spec_from_file_location('rhu_classifier_api',api_path)
api=importlib.util.module_from_spec(spec)
spec.loader.exec_module(api)
reloaded=api.load_bundle(model_path)
probe=list(test.question_text.head(30))+['', '   ', 'What are the prerequisites for COSC 351?']
results=api.predict_questions(reloaded,probe)
assert all(x['routing_status']=='invalid_input' for x in results[-3:-1])
assert results[-1]['question_text']=='What are the prerequisites for COSC 351?'
for k in range(30):
    assert results[k]['predicted_category']==final_cat[k]
    assert results[k]['predicted_intent']==final_int[k]
assert all(x['predicted_intent'] is None or final_map[x['predicted_intent']]==x['predicted_category'] for x in results)
for pred in results:
    qa = api.make_qa_request(pred)
    assert qa['original_question'] == pred['question_text']
    assert qa['category_hard_filter'] is False and qa['intent_hard_filter'] is False
    assert qa['requires_source_citation'] is True and qa['fallback_if_no_evidence'] is True
    assert qa['retrieve_from_full_catalog'] is (pred['routing_status']!='invalid_input')
assert api.make_qa_request(results[-1])['original_question'].endswith('COSC 351?')
print('Export/reload/inference and QA-handoff tests: PASS')
print('Sample raw question ->',results[-1])
print('Saved model:',model_path, '| size KB:',round(model_path.stat().st_size/1024))

Export/reload/inference and QA-handoff tests: PASS
Sample raw question -> {'question_text': 'What are the prerequisites for COSC 351?', 'predicted_category': 'courses', 'predicted_intent': 'course_prerequisite', 'raw_category': 'courses', 'raw_intent': 'course_prerequisite', 'routing_status': 'ok', 'review_needed': False}
Saved model: rhu_v31_outputs/rhu_classifier_v31.joblib | size KB: 1450


## 11A. QA handoff checks and non-scored demonstration

Classification is **optional metadata for retrieval**, not a guardrail that decides whether a catalog answer exists. Pass the untouched student question to the search stage. Use predicted category/intent only as possible ranking boosts; search all catalog passages, verify supporting evidence, cite the source, and fall back when no evidence is found. Questions below are **smoke tests, not independently labeled evaluation data**. A classifier may confidently assign a category to an out-of-scope question.

In [14]:
sample_questions = [
    'What are the prerequisites for COSC 351?',
    'How many credits are required for the Computer Science degree?',
    'What is the tuition per credit?',
    'How do I register for next semester?',
    'Who should I contact about academic advising?',
    'Is COSC 351 offered next Tuesday?',
    'Who won the last football World Cup?',
    'I liked registration but disliked advising.',
    '',
]
smoke = api.predict_questions(reloaded,sample_questions)
smoke_rows = []
for pred in smoke:
    req = api.make_qa_request(pred)
    assert req['original_question'] == pred['question_text']
    assert req['category_hard_filter'] is False and req['intent_hard_filter'] is False
    assert req['requires_verified_evidence'] and req['requires_source_citation']
    assert pred['predicted_intent'] is None or final_map[pred['predicted_intent']] == pred['predicted_category']
    smoke_rows.append({'question':req['original_question'],'category_hint':req['category_hint'],
       'intent_hint':req['intent_hint'],'routing_status':req['routing_status'],
       'retrieval_eligible':req['retrieve_from_full_catalog']})
smoke_df = pd.DataFrame(smoke_rows)
smoke_df.to_csv(OUT_DIR/'qa_handoff_smoke_unlabeled.csv',index=False)
display(smoke_df)
print('QA handoff contract tests: PASS. Not an independent accuracy evaluation.')

                                            question  ... retrieval_eligible
0           What are the prerequisites for COSC 351?  ...               True
1  How many credits are required for the Computer...  ...               True
2                    What is the tuition per credit?  ...               True
3               How do I register for next semester?  ...               True
4      Who should I contact about academic advising?  ...               True
5                  Is COSC 351 offered next Tuesday?  ...               True
6               Who won the last football World Cup?  ...               True
7        I liked registration but disliked advising.  ...               True
8                                                     ...              False

[9 rows x 5 columns]
QA handoff contract tests: PASS. Not an independent accuracy evaluation.


## 12. Package, document, and audit the V3 result

The output ZIP contains the model bundle, source-only inference helper, evaluation tables, an empty independently authored-question template, and the environment manifest. It does not include source datasets or the notebook itself, which is distributed separately. For use in a chatbot, **do not** hard-filter retrieved catalog passages based solely on these predictions: boost category matches, but allow other relevant evidence and return a cited answer or an explicit fallback.

In [15]:
import zipfile
report = {
  'notebook_version':'V3.1 FINAL','validation_grid_source':grid_source,'source':str(DATA_SOURCE),'source_type':source_name,
  'input_sha256':source_hashes,
  'rows_after_exclusions':{s:len(clean[s]) for s in clean},
  'taxonomy_source_audit':taxonomy_audit_status,
  'source_taxonomy_multi_parent_labels':multi_parent_source_labels,
  'trained_taxonomy_manifest':'documented_trained_intent_taxonomy.csv',
  'qa_handoff_contract':'make_qa_request() exports soft hints only, never hard filtering',
  'trained_targets':{target:sorted(trainval[target].unique().tolist()) for target in TARGETS},
  'validation_selected_family':selected_families,'validation_selected_routing_policy':SELECTED_POLICY,
  'macroF1_definition':'gold-present labels and known gold labels reported separately; unresolved counted incorrect',
  'historical_test_previously_inspected':True,
  'test_metrics':json.loads(metrics_df.to_json(orient='records')),
  'routing_validation_table':json.loads(policy_table.to_json(orient='records')),
  'test_intent_unseen_labels':sorted(set(test.intent)-set(trainval.intent)),
  'test_intent_unseen_rows':int((~test.intent.isin(set(trainval.intent))).sum()),
  'train_intents_single_group':int((train.groupby('intent').group_id.nunique()==1).sum()),
  'reconciled_test_pair_invalid_count':int(sum(i is not None and final_map.get(i)!=c for c,i in zip(final_cat,final_int))),
  'routed_test_unresolved_intent_rows':int(pd.isna(final_int).sum()),
  'independent_student_test_run':bool(STUDENT_FILE.exists()),
  'limitations':['Original held-out test inspected before V3','Synthetic/template phrasing is not student speech',
     'Insufficient independent groups and missing training intent labels','No validated unknown-input abstention threshold',
     'Classification does not establish grounded answer correctness or retrieval quality']}
(OUT_DIR/'v3_experiment_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
reqs = '\n'.join(['# Tested environment (verify compatibility when loading joblib across environments)',
                  f'scikit-learn=={sklearn.__version__}',f'numpy=={np.__version__}',
                  f'pandas=={pd.__version__}',f'joblib=={joblib.__version__}', ''])
(OUT_DIR/'requirements_v31.txt').write_text(reqs,encoding='utf-8')
readme = f"""# RHU Question Classification V3.1 final capstone package

Selected category model: {selected_families['category']}
Selected intent model: {selected_families['intent']}
Selected validation-only reconciliation: {SELECTED_POLICY}

## Use
Place rhu_classifier_v31.joblib and rhu_classifier_api.py together. Install dependencies from requirements_v31.txt, then:

```python
from rhu_classifier_api import load_bundle, predict_one
model = load_bundle('rhu_classifier_v31.joblib')
result = predict_one(model, 'What are the prerequisites for COSC 351?')
print(result)
from rhu_classifier_api import make_qa_request
print(make_qa_request(result))
```

Use `question_text` unchanged for retrieval. Pass `make_qa_request(result)` to the retrieval module; hints MUST NOT hard filter passages.
The taxonomy manifest lists precisely the trained intent-to-category mapping, audited against the original release taxonomy. The source release contains a multi-parent descriptive intent name that is not among the trained labels.

**One-month capstone decision:** freeze the model here, implement grounded QA with page citations, and then evaluate retrieval and end-to-end chatbot behavior. Do not tune on the already-inspected historical test. `review_needed` flags routing/input problems, NOT all unknown facts. This classifier does not verify catalog answers. Do not treat SVM margins as probabilities.

## Honest interpretation
The historical test was reviewed in V1/V2; V3 must not call it untouched.
All original splits are preserved. No test rows were moved into training. The project still needs a new human-labeled independent student-question evaluation. Never treat the empty student template as an evaluation dataset.
The category/intent taxonomy is based on the train+validation split. No silent relabeling or guessing unseen classes.
"""
(OUT_DIR/'README_model.md').write_text(readme,encoding='utf-8')
package_path=Path('RHU_Question_Classification_V31_Final_Model_Package.zip')
with zipfile.ZipFile(package_path,'w',zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file() and path.suffix in ('.py','.joblib','.txt','.json','.csv','.md'):
            z.write(path,arcname=path.name)
print('Packaged:',package_path,'| artifacts:',len(zipfile.ZipFile(package_path).namelist()))
print('\nFINAL SUMMARY')
print(metrics_df[['stage','target','accuracy_all','macroF1_gold_present','macroF1_known_gold','unknown_rows']].round(4).to_string(index=False))
print('Selected validation policy:',SELECTED_POLICY,'| Total elapsed %.1fs'%(time.perf_counter()-T0))

Packaged: RHU_Question_Classification_V31_Final_Model_Package.zip | artifacts: 14

FINAL SUMMARY
 stage   target  accuracy_all  macroF1_gold_present  macroF1_known_gold  unknown_rows
   raw category        0.8175                0.4801              0.4801             0
   raw   intent        0.7537                0.4198              0.8405           324
routed category        0.8175                0.4801              0.4801             0
routed   intent        0.7496                0.3979              0.8082           324
Selected validation policy: category_governs | Total elapsed 28.0s


## 13. Evaluation and decision (read with printed results)

**Acceptance conditions**

* All cells execute successfully with the provided dataset and produce an independently loadable inference API.
* There are **zero contradictory category/intent pairs among resolved predictions**. When the rules cannot resolve a pair, return a missing intent and a review flag rather than inventing evidence.
* Train, validation, test group IDs and question IDs remain disjoint; **no test examples** are added to training.
* The report includes known-label and all-gold-label macro F1, unsupported labels, one-group labels, group-level accuracy, and the non-template subset. Accuracy need not increase; factual evaluation quality matters more than optimizing a previously seen test.
* The optional external test is honestly marked *not performed* unless a real labeled dataset is provided.

**Remaining work outside this notebook:** add independent questions per desired class, adjudicate the label taxonomy, evaluate unknown/out-of-catalog questions and spoken transcription errors, and test whether the classifier genuinely improves grounded retrieval over retrieval without classification. The larger project also requires ABSA, the voice interface, and end-to-end QA evaluation.

**V3.1 final integration gate:** The trained mapping is written to `documented_trained_intent_taxonomy.csv` and cross-checked against the original label taxonomy where available. `make_qa_request()` enforces a soft-routing-only interface. No new model was selected using historical test data. The classification phase can close for this one-month prototype; the next milestone is source-grounded retrieval with page references and explicit no-evidence fallback.